# Day 7 | Hands-On 1: Verify Dimensions + Bridge Table Concept
### GlobalMart Data Engineering Bootcamp

| | |
|---|---|
| **Follows** | Day 6 HOL 1 — all 6 dimensions already built in `<your-catalog>.gold` |
| **Source** | `<your-catalog>.gold.dim_*` (verify), `<your-catalog>.gold.dim_product` (bridge example) |
| **Target** | `<your-catalog>.gold.dim_campaign`, `<your-catalog>.gold.bridge_product_campaign` (new, illustrative) |
| **Duration** | 60 minutes |

### Learning Objectives
- Confirm yesterday's 6 dimensions are still healthy before building on top of them
- Understand what a bridge table is and exactly when a star schema needs one
- Decide, with a real justification, whether GlobalMart's core model needs one today
- Build one small illustrative bridge table so the concept isn't purely theoretical

---
**Instructions:** Run each cell with **Shift + Enter**. Phase A is a 2-minute sanity check, not new work — don't skip it, since everything downstream (today and Day 7 HOL 2) assumes these 6 tables are already correct. (Code cells below use the literal `gbmart` catalog — GlobalMart's real Gold-layer run; your own catalog will be named differently.)

---
## Phase A — Verify Day 6's Dimensions Are Still Healthy

In [ ]:
# ============================================================
# CELL 1: Re-verify all 6 dimensions from Day 6
#
# This is the exact same check Day 6 ended on. If any table is
# missing or shows 0 rows, go back and re-run the relevant cell
# in Day6_4_HOL1_Star_Schema_Design.ipynb before continuing --
# today's bridge-table work and Day 7 HOL 2's fact_sales build
# both assume these are correct right now.
# ============================================================

print(f"{'Table':<25} {'Rows':>10}  {'Columns':>8}")
print("-" * 48)

for table in ["dim_customer", "dim_product", "dim_date", "dim_address", "dim_payment_method", "dim_orders"]:
    df = spark.table(f"gbmart.gold.{table}")
    print(f"{table:<25} {df.count():>10,}  {len(df.columns):>8}")

---
## Phase B — What Is a Bridge Table, and When Do You Actually Need One?

Every dimension you built on Day 6 relates to `fact_sales` (which you'll build in HOL 2 today) with a clean **many-to-one** relationship: many order lines share one customer, many order lines share one product, and so on. A plain foreign key handles many-to-one perfectly — that's why `Customer_ID` on `fact_sales` is enough, no extra table needed. (One detail worth being precise about: in the real build, most of these IDs are carried straight from Silver rather than resolved through an actual join to the Gold `dim_*` table — `gold.dim_date` is the one real Gold-table join in the fact build. That's a separate design decision from *this* one; the many-to-one shape, and the "no bridge needed" conclusion, hold either way.)

A **bridge table** exists for a different shape: **many-to-many**. Classic textbook example — a bank account can have multiple joint holders, and a person can hold multiple accounts. Neither `dim_account` nor `dim_customer` can hold a single foreign key to the other, because there isn't one right answer. The fix is a third table sitting *between* the two, with one row per valid (account, holder) pair:

```
dim_customer  <---  bridge_account_holder  --->  dim_account
  customer_id         customer_id, account_id       account_id
                       (one row per pairing)
```

Anything joining fact data through this bridge has to be careful about **double-counting**: a $500 deposit touching 2 joint holders should not silently become $1,000 across the report.

---
## Phase C — Does GlobalMart's `fact_sales` Actually Need One?

**Check the grain first, always.** `fact_sales` (built next, in HOL 2) has grain = **one row per order line item** (`order_items`). Walk each relationship at that grain:

| Relationship at the `order_items` grain | Shape | Bridge needed? |
|---|---|---|
| order line → customer | many-to-one | No — plain FK |
| order line → product | many-to-one | No — plain FK |
| order line → order | many-to-one | No — plain FK |
| order line → shipping address | many-to-one (GlobalMart picks one primary address per customer) | No — plain FK |
| order line → payment | many-to-one (payments are 1:1 with orders in this model) | No — plain FK |

**Conclusion: GlobalMart's core `fact_sales` does not need a bridge table today.** This isn't a shortcut or an oversight — `order_items` being the grain is *exactly* what already resolves the one real many-to-many relationship in this data (an order can contain many products, and a product appears on many orders). The grain choice did the work; a bridge table would be solving a problem that doesn't exist here.

> **Where a real GlobalMart bridge table WOULD show up:** if GlobalMart started running marketing campaigns where **one product can be tagged under several campaigns at once, and one campaign covers many products** — that's a genuine many-to-many, and no single FK on either side can hold it. Build that below as a worked example.

---
## Phase D — Build the Illustrative Bridge: Products ↔ Campaigns

In [ ]:
# ============================================================
# CELL 2: dim_campaign -- a small new dimension, one row per campaign
#
# This is deliberately synthetic/illustrative data -- GlobalMart's
# real source systems (Postgres, ADLS) don't have a campaigns feed
# today. We're building this purely to make the bridge-table
# pattern concrete, not because it's part of the graded pipeline.
# ============================================================

from pyspark.sql.functions import sha2, col, lit

campaign_rows = [
    ("CMP-01", "Diwali Mega Sale",      "2025-10-15", "2025-11-05"),
    ("CMP-02", "New Year Electronics",   "2026-01-01", "2026-01-15"),
    ("CMP-03", "Summer Furniture Fest",  "2026-04-01", "2026-04-30"),
]

dim_campaign_df = spark.createDataFrame(
    campaign_rows, ["campaign_id", "campaign_name", "start_date", "end_date"]
).withColumn("campaign_sk", sha2(col("campaign_id"), 256))

dim_campaign_df.write.format("delta").mode("overwrite").saveAsTable("gbmart.gold.dim_campaign")
print(f"dim_campaign rows: {spark.table('gbmart.gold.dim_campaign').count()}")

In [ ]:
# ============================================================
# CELL 3: bridge_product_campaign -- one row per (product, campaign) pairing
#
# This is the bridge itself. Notice it has NO measures of its own --
# a bridge table is pure plumbing, connecting two dimensions that
# can't connect directly through fact_sales.
# ============================================================

# Pick a handful of real product_ids from dim_product to tag against campaigns,
# so this bridge joins against real data, not more synthetic keys.
sample_products = [row.product_id for row in
    spark.table("gbmart.gold.dim_product").filter("is_current = true").select("product_id").limit(6).collect()]

print(f"Sample product_ids used for the bridge: {sample_products}")

# Deliberately overlapping: product[0] is in two campaigns, campaign CMP-01
# covers three products -- this IS the many-to-many the bridge exists for.
bridge_rows = [
    (sample_products[0], "CMP-01"),
    (sample_products[0], "CMP-02"),   # same product, second campaign
    (sample_products[1], "CMP-01"),
    (sample_products[2], "CMP-01"),
    (sample_products[3], "CMP-02"),
    (sample_products[4], "CMP-03"),
]

bridge_df = spark.createDataFrame(bridge_rows, ["product_id", "campaign_id"])
bridge_df.write.format("delta").mode("overwrite").saveAsTable("gbmart.gold.bridge_product_campaign")
print(f"bridge_product_campaign rows: {spark.table('gbmart.gold.bridge_product_campaign').count()}")

---
## Phase E — Prove the Double-Counting Risk, Then Query It Correctly

In [ ]:
# ============================================================
# CELL 4: This is exactly the mistake a bridge table can cause if
# you forget it's many-to-many -- joining fact_sales through the
# bridge multiplies rows for any product tagged under >1 campaign.
# ============================================================

print("Rows in dim_product BEFORE joining through the bridge:")
print(spark.table("gbmart.gold.dim_product").filter("is_current = true").count())

joined = (
    spark.table("gbmart.gold.dim_product").filter("is_current = true")
    .join(spark.table("gbmart.gold.bridge_product_campaign"), "product_id")
)
print("Rows AFTER joining through the bridge (note: product[0] now appears twice):")
joined.select("product_id", "product_name", "campaign_id").orderBy("product_id").display()

In [ ]:
# ============================================================
# CELL 5: The safe version -- if you need a SALES number attributed
# per campaign, join fact_sales -> bridge -> dim_campaign and be
# explicit that a multi-campaign product's sales get counted once
# per campaign it belongs to (that's a deliberate business decision,
# not a bug -- a sale genuinely counts toward every campaign that
# was running for that product).
# ============================================================

spark.sql("""
    SELECT
        c.campaign_name,
        COUNT(DISTINCT f.Order_ID)   AS orders_touching_campaign,
        SUM(f.Sales_amount)         AS attributed_sales
    FROM gbmart.gold.fact_sales f
    JOIN gbmart.gold.bridge_product_campaign b ON f.Product_ID = b.product_id
    JOIN gbmart.gold.dim_campaign c            ON b.campaign_id = c.campaign_id
    GROUP BY c.campaign_name
    ORDER BY attributed_sales DESC
""").display()

print("NOTE: this cell reads gbmart.gold.fact_sales, which you build in HOL 2 later today.")
print("If you're running this notebook before HOL 2, come back to this cell afterward.")

---
## Key Takeaways

1. **A bridge table exists for many-to-many relationships only** — if a plain foreign key can express the relationship, you don't need one.
2. **Check the grain before reaching for a bridge.** GlobalMart's `order_items` grain already resolves the order↔product many-to-many — that's not a coincidence, it's the grain doing its job.
3. **A bridge table carries no measures of its own** — it's pure connective plumbing between two dimensions.
4. **Joining through a bridge can silently multiply rows.** Always check row counts before/after, and be explicit (in the query and in the report) about how multi-way attribution is being counted.

### Submission Checklist
- [ ] Phase A verification shows all 6 Day 6 dimensions healthy
- [ ] `dim_campaign` and `bridge_product_campaign` created successfully
- [ ] Can explain, in one sentence, why `fact_sales` itself doesn't need a bridge table
- [ ] Ran the double-counting demo (Cell 4) and the correct attribution query (Cell 5)

---
## Reset (if needed)

In [ ]:
# spark.sql("DROP TABLE IF EXISTS gbmart.gold.bridge_product_campaign")
# spark.sql("DROP TABLE IF EXISTS gbmart.gold.dim_campaign")
# print("Illustrative bridge tables dropped -- the 6 real dimensions from Day 6 are untouched")